In [2]:
import pyodbc

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=localhost\SQLEXPRESS;DATABASE=master;Trusted_Connection=yes;",
    autocommit=True
)
cur = conn.cursor()
cur.execute("CREATE DATABASE BA_Airline_Commercial_Analytics")
conn.close()

conn2 = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=localhost\SQLEXPRESS;DATABASE=BA_Airline_Commercial_Analytics;Trusted_Connection=yes;",
    autocommit=True
)
cur2 = conn2.cursor()
for schema in ['bronze', 'silver', 'gold']:
    cur2.execute(f"CREATE SCHEMA {schema}")
conn2.close()
print("database and schemas created")

database and schemas created


#### Load Patil flight-delay data into bronze. 
#### Reads the 7M-row csv in chunks of 200,000 rows at a time (loading it all at once would oveload  memory) and appends each chunk into a new SQL Server table, bronze.patil_flights. 

#### This is the raw, untouched landing zone - no cleaning happens here.







In [5]:
import pandas as pd
from sqlalchemy import create_engine, URL

conn_url = URL.create(
    "mssql+pyodbc", host=r"localhost\SQLEXPRESS",
    database="BA_Airline_Commercial_Analytics",
    query={"driver": "ODBC Driver 17 for SQL Server", "trusted_connection": "yes"},
)
engine = create_engine(conn_url)

for i, chunk in enumerate(pd.read_csv(
    r"C:\Projects\BA_Airline_Commercial_Analytics\raw_data\patil_delays_2024\flight_data_2024.csv",
    chunksize=200_000
)):
    chunk.to_sql("patil_flights", engine, schema="bronze",
                  if_exists="append" if i else "replace", index=False)
    print(f"chunk {i} loaded")

print("patil done")

C:\Users\HARDEEP\anaconda3\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


chunk 0 loaded
chunk 1 loaded
chunk 2 loaded


C:\Users\HARDEEP\AppData\Local\Temp\ipykernel_13908\2537080561.py:11: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(


chunk 3 loaded
chunk 4 loaded
chunk 5 loaded
chunk 6 loaded
chunk 7 loaded
chunk 8 loaded
chunk 9 loaded
chunk 10 loaded
chunk 11 loaded
chunk 12 loaded
chunk 13 loaded
chunk 14 loaded
chunk 15 loaded
chunk 16 loaded
chunk 17 loaded
chunk 18 loaded
chunk 19 loaded
chunk 20 loaded
chunk 21 loaded
chunk 22 loaded
chunk 23 loaded
chunk 24 loaded
chunk 25 loaded
chunk 26 loaded
chunk 27 loaded
chunk 28 loaded
chunk 29 loaded
chunk 30 loaded


C:\Users\HARDEEP\AppData\Local\Temp\ipykernel_13908\2537080561.py:11: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(


chunk 31 loaded
chunk 32 loaded


C:\Users\HARDEEP\AppData\Local\Temp\ipykernel_13908\2537080561.py:11: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(


chunk 33 loaded
chunk 34 loaded
chunk 35 loaded
patil done


#### Loading fares dataset into bronze
#### Small enough to load in one go, no chunking needed. This lands raw and
#### untouched in bronze.amitzala_fares, same as Patil.

In [6]:
pd.read_csv(
    r"C:\Projects\BA_Airline_Commercial_Analytics\raw_data\amitzala_db1b_fares\US Airline Flight Routes and Fares.csv"
).to_sql("amitzala_fares", engine, schema="bronze", if_exists="replace", index=False)

print("amitzala done")

C:\Users\HARDEEP\AppData\Local\Temp\ipykernel_13908\2911720347.py:1: DtypeWarning: Columns (20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(


amitzala done


### Confirm row counts
#### Quick sanity check that both tables loaded with the expected number of rows.

In [7]:
from sqlalchemy import text

with engine.connect() as conn:
    for t in ["patil_flights", "amitzala_fares"]:
        count = conn.execute(text(f"SELECT COUNT(*) FROM bronze.{t}")).scalar()
        print(t, count)

patil_flights 7079081
amitzala_fares 245955


#### Build the silver-layer view: one row per route
#### Collapses 7M flight-level rows down to one row per origin-destination
#### pair — matching other dataset grain so the two can be compared. Built as a SQL
#### view (not a new table) so it always reflects the current bronze data.

In [8]:
with engine.connect() as conn:
    conn.execute(text("""
        CREATE OR ALTER VIEW silver.vw_PatilRouteAgg AS
        SELECT
            origin,
            dest,
            COUNT(*) AS total_flights,
            AVG(CAST(arr_delay AS FLOAT)) AS avg_arr_delay,
            SUM(cancelled) AS cancellations,
            AVG(CAST(distance AS FLOAT)) AS avg_distance
        FROM bronze.patil_flights
        GROUP BY origin, dest
    """))
    conn.commit()

print("silver view created")

silver view created


#### Renamed bronze tables to describe the data, not the source
#### Renamed away from dataset-author names (patil, amitzala) to names that
#### describe what the data actually contains.

In [9]:
with engine.connect() as conn:
    conn.execute(text("EXEC sp_rename 'bronze.patil_flights', 'flight_operations'"))
    conn.execute(text("EXEC sp_rename 'bronze.amitzala_fares', 'route_market_fares'"))
    conn.commit()

print("tables renamed")

tables renamed


#### Build the silver-layer view: one row per route. Collapses flight_operations down to one row per origin-destination pair —
#### matching route_market_fares' grain so the two can be compared.

#### Fix the silver view name and rebuild it cleanly
#### Dropping the old-named view and recreating it under the new naming
#### convention, pointing at the renamed bronze table.


In [10]:
with engine.connect() as conn:
    conn.execute(text("DROP VIEW IF EXISTS silver.vw_PatilRouteAgg"))
    conn.execute(text("""
        CREATE OR ALTER VIEW silver.vw_RoutePerformance AS
        SELECT
            origin,
            dest,
            COUNT(*) AS total_flights,
            AVG(CAST(arr_delay AS FLOAT)) AS avg_arr_delay,
            SUM(cancelled) AS cancellations,
            AVG(CAST(distance AS FLOAT)) AS avg_distance
        FROM bronze.flight_operations
        GROUP BY origin, dest
    """))
    conn.commit()

print("view fixed")

view fixed


#### Verify the silver view works
#### Pulls a few rows from the new view to confirm it's returning sensible,
#### route-level aggregated data.

In [11]:
check = pd.read_sql("SELECT TOP 10 * FROM silver.vw_RoutePerformance", engine)
print(check)

  origin dest  total_flights  avg_arr_delay  cancellations  avg_distance
0    ATL  ABE           1017       1.664000             14         692.0
1    BNA  ABE            105       0.355769              1         685.0
2    CLT  ABE           1166      12.866492             13         481.0
3    DEN  ABE             86       9.082353              1        1539.0
4    FLL  ABE             81       1.604938              0        1041.0
5    MCO  ABE             68      16.772727              2         906.0
6    MLB  ABE            105      -9.106796              2         914.0
7    MYR  ABE            188      30.551351              3         518.0
8    ORD  ABE            314      20.571906             13         655.0
9    PGD  ABE            197      10.309278              3        1018.0


#### Check the route match rate between operations and fares data
#### Before joining, measure how many routes in flight_operations actually have
#### a corresponding entry in route_market_fares. This determines whether an
#### inner join is viable and how much of the network it will cover.

In [12]:
match_check = pd.read_sql("""
    SELECT
        COUNT(DISTINCT p.origin + p.dest) AS total_routes,
        COUNT(DISTINCT CASE WHEN a.airport_1 IS NOT NULL
              THEN p.origin + p.dest END) AS matched_routes
    FROM silver.vw_RoutePerformance p
    LEFT JOIN bronze.route_market_fares a
        ON p.origin = a.airport_1 AND p.dest = a.airport_2
""", engine)

print(match_check)

   total_routes  matched_routes
0          6805            1820


#### Sanity check: raw preview of both bronze tables
#### Quick look at 10 rows from each source table before building the gold join,
#### just to confirm the data looks the way we expect.

In [15]:
pd.set_option('display.max_columns', None)

print("=== flight_operations (first 10 rows) ===")
ops_preview = pd.read_sql("SELECT TOP 10 * FROM bronze.flight_operations", engine)
print(ops_preview)

print("\n=== route_market_fares (first 10 rows) ===")
fares_preview = pd.read_sql("SELECT TOP 10 * FROM bronze.route_market_fares", engine)
print(fares_preview)

=== flight_operations (first 10 rows) ===
   year  month  day_of_month  day_of_week     fl_date op_unique_carrier  \
0  2024      1             1            1  2024-01-01                9E   
1  2024      1             1            1  2024-01-01                9E   
2  2024      1             1            1  2024-01-01                9E   
3  2024      1             1            1  2024-01-01                9E   
4  2024      1             1            1  2024-01-01                9E   
5  2024      1             1            1  2024-01-01                9E   
6  2024      1             1            1  2024-01-01                9E   
7  2024      1             1            1  2024-01-01                9E   
8  2024      1             1            1  2024-01-01                9E   
9  2024      1             1            1  2024-01-01                9E   

   op_carrier_fl_num origin     origin_city_name origin_state_nm dest  \
0             4814.0    JFK         New York, NY        New

#### Build the gold-layer view: joined route performance + competitive fares
#### Inner join between operations and fares data on the 1,820 routes that exist
#### in both sources. This is the actual analysis-ready table — one row per
#### route, carrying both operational and competitive metrics side by side.

In [16]:
with engine.connect() as conn:
    conn.execute(text("""
        CREATE OR ALTER VIEW gold.vw_RouteCompetitivePerformance AS
        SELECT
            p.origin,
            p.dest,
            p.total_flights,
            p.avg_arr_delay,
            p.cancellations,
            p.avg_distance,
            a.carrier_lg,
            a.fare_lg,
            a.large_ms,
            a.carrier_low,
            a.fare_low,
            a.lf_ms,
            a.passengers
        FROM silver.vw_RoutePerformance p
        INNER JOIN bronze.route_market_fares a
            ON p.origin = a.airport_1 AND p.dest = a.airport_2
    """))
    conn.commit()

print("gold view created")

gold view created


#### Preview the gold-layer joined table
#### First real look at the combined operations + competitive fare data,
#### route by route.

In [17]:
gold_preview = pd.read_sql("SELECT TOP 10 * FROM gold.vw_RouteCompetitivePerformance", engine)
print(gold_preview)

print("\ntotal rows in gold view:")
count = pd.read_sql("SELECT COUNT(*) AS row_count FROM gold.vw_RouteCompetitivePerformance", engine)
print(count)

  origin dest  total_flights  avg_arr_delay  cancellations  avg_distance  \
0    ABE  PIE            261      -0.822835              6         970.0   
1    ABQ  DAL           1695       5.278640             16         580.0   
2    ABQ  DFW           2460      19.347589             66         569.0   
3    ABQ  PHX           3243       3.925170              6         328.0   
4    ABQ  BWI            396      11.173469              2        1670.0   
5    ABQ  MDW            505       0.141153              0        1121.0   
6    ABQ  ORD            828       2.212195              5        1118.0   
7    ABQ  HOU            962       3.342495              9         759.0   
8    ABQ  IAH            508       2.744939             13         744.0   
9    ABQ  JFK            259      44.326772              5        1826.0   

  carrier_lg  fare_lg  large_ms carrier_low  fare_low   lf_ms  passengers  
0         G4    81.43    1.0000          G4     81.43  1.0000         180  
1         W

#### Diagnose the row duplication
#### Checking how many fare rows exist per route in route_market_fares — this
#### should reveal why the join produced far more rows than the match count.

In [18]:
dupe_check = pd.read_sql("""
    SELECT airport_1, airport_2, COUNT(*) AS row_count
    FROM bronze.route_market_fares
    GROUP BY airport_1, airport_2
    ORDER BY row_count DESC
""", engine)

print(dupe_check.head(10))
print("\nmax rows for a single route:", dupe_check['row_count'].max())
print("average rows per route:", dupe_check['row_count'].mean())

  airport_1 airport_2  row_count
0       MDW       BDL        118
1       ORD       BDL        118
2       BOS       BNA        118
3       BUR       BNA        118
4       CAK       BNA        118
5       CLE       BNA        118
6       DFW       BNA        118
7       FLL       BNA        118
8       HOU       BNA        118
9       IAH       BNA        118

max rows for a single route: 118
average rows per route: 60.44605554190219


#### Check the actual year range in route_market_fares
#### The row-per-route duplication suggests this covers many years of quarterly
#### data, not just Q3 2021 as originally seen in the sample. Confirming the
#### real range before deciding how to aggregate.


In [19]:
year_range = pd.read_sql("""
    SELECT MIN(Year) AS earliest_year, MAX(Year) AS latest_year,
           COUNT(DISTINCT Year) AS distinct_years,
           COUNT(DISTINCT CONCAT(Year, '-', quarter)) AS distinct_year_quarters
    FROM bronze.route_market_fares
""", engine)

print(year_range)

   earliest_year  latest_year  distinct_years  distinct_year_quarters
0           1993         2024              31                     118


#### Rebuild gold view: filter to 2024, average across quarters
#### The fares table spans 1993-2024. Filtering to 2024 only, matching the
#### operations data's year, and averaging across whatever quarters exist within
#### 2024 so each route still gets exactly one row.

In [20]:
with engine.connect() as conn:
    conn.execute(text("""
        CREATE OR ALTER VIEW silver.vw_RouteFares2024 AS
        SELECT
            airport_1,
            airport_2,
            AVG(fare_lg) AS avg_fare_lg,
            AVG(large_ms) AS avg_large_ms,
            AVG(fare_low) AS avg_fare_low,
            AVG(lf_ms) AS avg_lf_ms,
            SUM(passengers) AS total_passengers
        FROM bronze.route_market_fares
        WHERE Year = 2024
        GROUP BY airport_1, airport_2
    """))

    conn.execute(text("""
        CREATE OR ALTER VIEW gold.vw_RouteCompetitivePerformance AS
        SELECT
            p.origin,
            p.dest,
            p.total_flights,
            p.avg_arr_delay,
            p.cancellations,
            p.avg_distance,
            f.avg_fare_lg,
            f.avg_large_ms,
            f.avg_fare_low,
            f.avg_lf_ms,
            f.total_passengers
        FROM silver.vw_RoutePerformance p
        INNER JOIN silver.vw_RouteFares2024 f
            ON p.origin = f.airport_1 AND p.dest = f.airport_2
    """))
    conn.commit()

print("gold view rebuilt with 2024-only fares")

gold view rebuilt with 2024-only fares


#### Confirm the duplication is fixed
#### Row count should now be close to the actual number of matched routes,
#### not inflated by repeated fare-quarter rows.

In [22]:
count = pd.read_sql("SELECT COUNT(*) AS row_count FROM gold.vw_RouteCompetitivePerformance", engine)
print(count)

preview = pd.read_sql("SELECT TOP 10 * FROM gold.vw_RouteCompetitivePerformance", engine)
print(preview)

   row_count
0       1274
  origin dest  total_flights  avg_arr_delay  cancellations  avg_distance  \
0    ATW  AZA            198      10.367347              2        1452.0   
1    BIS  AZA            283      -4.562724              3        1092.0   
2    BOI  AZA            103      12.843137              1         749.0   
3    BZN  AZA            129       1.335938              1         861.0   
4    CID  AZA            204      -3.433498              1        1240.0   
5    CVG  AZA             80      14.337500              0        1553.0   
6    DSM  AZA            180      14.337079              2        1138.0   
7    EUG  AZA            121      17.025641              4         971.0   
8    FAR  AZA            333      -2.870871              0        1220.0   
9    GRR  AZA            194      -5.803109              1        1561.0   

   avg_fare_lg  avg_large_ms  avg_fare_low  avg_lf_ms  total_passengers  
0       168.30           1.0        168.30        1.0          

#### Separate view: multi-year fare trends (amitzala only, no join)
#### This is a standalone view for later forecasting work — average fare and
#### market concentration per route per year, across the full 1993-2024 history.
#### Not joined to operations data, since that only spans 2024.

In [1]:
with engine.connect() as conn:
    conn.execute(text("""
        CREATE OR ALTER VIEW silver.vw_RouteFareTrends AS
        SELECT
            airport_1,
            airport_2,
            Year,
            AVG(fare_lg) AS avg_fare_lg,
            AVG(large_ms) AS avg_large_ms,
            AVG(fare_low) AS avg_fare_low,
            SUM(passengers) AS total_passengers
        FROM bronze.route_market_fares
        GROUP BY airport_1, airport_2, Year
    """))
    conn.commit()

print("fare trends view created")

NameError: name 'engine' is not defined